In [0]:
# Create sample data
data = [
    (1, "ABHAY", 95),
    (2, "BANTY", 90),
    (3, "AMAN", 98)
]

columns = ["student_id", "name", "score"]

df = spark.createDataFrame(data, columns)
df.show()


+----------+-----+-----+
|student_id| name|score|
+----------+-----+-----+
|         1|ABHAY|   95|
|         2|BANTY|   90|
|         3| AMAN|   98|
+----------+-----+-----+



In [0]:
# CREATE DELTA TABLE
spark.sql("CREATE TABLE IF NOT EXISTS students (student_id LONG, name STRING, score LONG) USING DELTA")

DataFrame[]

In [0]:
# Save as Delta table
df.write.format("delta").mode("overwrite").saveAsTable("students") # Overall path -  workspace.default.students

# Read back
delta_df = spark.read.format("delta").table("students")
delta_df.show()


+----------+-----+-----+
|student_id| name|score|
+----------+-----+-----+
|         1|ABHAY|   95|
|         2|BANTY|   90|
|         3| AMAN|   98|
+----------+-----+-----+



In [0]:
from delta.tables import DeltaTable

# Load Delta table
delta_table = DeltaTable.forName(spark, "workspace.default.students")

# Update score for ABHAY
delta_table.update(
    condition = "name = 'ABHAY'",
    set = { "score": "95" }
)

# Show updated data
delta_table.toDF().show()


+----------+-----+-----+
|student_id| name|score|
+----------+-----+-----+
|         1|ABHAY|   95|
|         2|BANTY|   90|
|         3| AMAN|   98|
+----------+-----+-----+



In [0]:
new_data = [(4, "David", 88)]
new_df = spark.createDataFrame(new_data, columns)

new_df.write.format("delta").mode("append").saveAsTable("students")

# Show current table
spark.read.format("delta").table("students").show()


+----------+-----+-----+
|student_id| name|score|
+----------+-----+-----+
|         2|BANTY|   90|
|         3| AMAN|   98|
|         1|ABHAY|   95|
|         4|David|   88|
+----------+-----+-----+



In [0]:
# Check history
delta_table.history().show()

+-------+-------------------+--------------+--------------------+--------------------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|        userId|            userName|           operation| operationParameters| job|          notebook|queryHistoryStatementId|           clusterId|readVersion|   isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+--------------+--------------------+--------------------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|      4|2026-10-09 14:09:28|75859355540436|abhay.aman.pandey...|               WRITE|{mode -> Append, ...|NULL|{2290081701894926}|   51f6b255-05f9-46d...|1009-134213-4hlt0

In [0]:
# Read version 1 (original data)
spark.read.format("delta").option("versionAsOf", 1).table("workspace.default.students").show()

# Read version 3 (after ABHAY update)
spark.read.format("delta").option("versionAsOf", 3).table("workspace.default.students").show()


+----------+-----+-----+
|student_id| name|score|
+----------+-----+-----+
|         1|ABHAY|   95|
|         2|BANTY|   90|
|         3| AMAN|   98|
+----------+-----+-----+

+----------+-----+-----+
|student_id| name|score|
+----------+-----+-----+
|         2|BANTY|   90|
|         3| AMAN|   98|
|         1|ABHAY|   95|
+----------+-----+-----+

